# 🥊 Motion Controller - Advanced Model Training

This notebook provides a comprehensive pipeline for training, validatin, and optimizing the motion controller model.

**Features:**
- Data Loading & validation
- Exploratory Data Analysis (EDA)
- Hyperparameter Tuning (GridSearchCV)
- Cross-Validation
- Feature Importance Analysis
- Learning Curves

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, learning_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Add parent director to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import src.config as config

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 📥 Load & Inspect Data

In [ ]:
data_dir = os.path.join(config.DATASET_DIR, "by_class")
dfs = []

print(f"Searching for data in: {data_dir}")

if os.path.exists(data_dir):
    csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    for file in csv_files:
        file_path = os.path.join(data_dir, file)
        try:
            df_part = pd.read_csv(file_path)
            dfs.append(df_part)
            print(f"  ✓ Loaded {file}: {len(df_part)} samples")
        except Exception as e:
            print(f"  ❌ Error loading {file}: {e}")
else:
    print("⚠ Split data directory not found. Trying raw file...")
    if os.path.exists(config.RAW_DATA_FILE):
        dfs.append(pd.read_csv(config.RAW_DATA_FILE))

if not dfs:
    raise FileNotFoundError("No data found! Please run src/data/collection.py first.")

df = pd.concat(dfs, ignore_index=True)
print(f"\n📊 Total Dataset Size: {len(df)} samples")
print(f"🔎 features per sample: {len(df.columns) - 1}")
df.sample(5)

## 2. 📊 Exploratory Data Analysis (EDA)
Checking class balance is crucial for a robust model.

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(x='label', data=df, palette='viridis')
plt.title('Class Distribution')
plt.xlabel('Pose Class')
plt.ylabel('Count')
plt.xticks(rotation=45)

for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='baseline', fontsize=11, color='black', xytext=(0, 5), 
                textcoords='offset points')
plt.show()

## 3. 🛠 Data Preprocessing & Splitting

**Strategy: Train-Validation-Test**

We use a **80-20 Split** combined with **Cross-Validation**:
1.  **Training Set (80%)**: Used to train the model.
    - *Validation*: We use **Cross-Validation (5-Folds)** within this set to tune hyperparameters. This automatically creates temporary Validation sets during training.
2.  **Test Set (20%)**: Held out completely. Used ONLY for the final evaluation to simulate real-world performance.

This method is more robust than a simple 3-way split for smaller datasets.

In [ ]:
X = df.iloc[:, :-1].values
y = df['label'].values

# Handle Missing Values
if np.isnan(X).any():
    print("⚠ NaN values detected! Replacing with 0.0")
    X = np.nan_to_num(X, nan=0.0)

# Label Encoding
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

print("Mapping:")
for idx, lbl in enumerate(encoder.classes_):
    print(f"  {idx} -> {lbl}")

# Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42,
    stratify=y_encoded
)

print("\nData Split Summary:")
print(f"🔹 Total Samples:    {len(X)}")
print(f"🔸 Training Set:     {len(X_train)} (80%) -> Used for Training & Cross-Validation")
print(f"🔸 Test Set:         {len(X_test)} (20%) -> Used for Final Evaluation")

## 4. 🧠 Feature Importance Analysis
Let's train a preliminary model to see which landmarks are most important for distinguishing poses.

In [ ]:
base_model = RandomForestClassifier(n_estimators=100, random_state=42)
base_model.fit(X_train, y_train)

importances = base_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Plot top 20 features
plt.figure(figsize=(15, 6))
plt.title("Top 20 Most Important Features")
plt.bar(range(20), importances[indices[:20]], align="center")
plt.xticks(range(20), indices[:20])
plt.xlabel("Feature Index (Landmark x/y/z/v)")
plt.ylabel("Importance")
plt.show()

## 5. 🎯 Hyperparameter Tuning (GridSearchCV)
We will search for the best model parameters automatically.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

print("Starting Grid Search... (This might take a moment)")
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, verbose=1, n_jobs=-1)
grid_search.fit(X_train, y_train)

print("\n✅ Best Parameters found:", grid_search.best_params_)
print(f"✅ Best Cross-Validation Accuracy: {grid_search.best_score_*100:.2f}%")

best_model = grid_search.best_estimator_

## 6. 📈 Model Evaluation
Running final evaluation on the held-out test set.

In [ ]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"🚀 Final Test Accuracy: {accuracy*100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=encoder.classes_))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('Confusion Matrix')
plt.show()

## 7. 🎓 Learning Curves
Checking if the model needs more data (High Variance vs High Bias).

In [ ]:
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X, y_encoded, cv=5, n_jobs=-1, 
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy'
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color="r", label="Training Score")
plt.plot(train_sizes, test_mean, 'o-', color="g", label="Cross-Validation Score")
plt.title("Learning Curve")
plt.xlabel("Training Examples")
plt.ylabel("Accuracy Score")
plt.legend(loc="best")
plt.grid(True)
plt.show()

## 8. 💾 Save Model

In [ ]:
config.ensure_directories()

with open(config.MODEL_FILE, 'wb') as f:
    pickle.dump(best_model, f)

with open(config.ENCODER_FILE, 'wb') as f:
    pickle.dump(encoder, f)
    
print(f"✅ Model saved to: {config.MODEL_FILE}")
print(f"✅ Encoder saved to: {config.ENCODER_FILE}")